# Show and Tell: Image Captioning Implementation
Modern implementation using TensorFlow 2.x and current best practices

In [ ]:
# Install required packages
!pip install tensorflow
!pip install nltk tqdm pandas matplotlib pycocotools
!pip install Pillow

In [ ]:
# Import necessary libraries
import tensorflow as tf
import numpy as np
import os
import pickle
import nltk
import json
import urllib.request
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import VGG16
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
nltk.download('punkt')
from google.colab import drive
from tqdm import tqdm

In [ ]:
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

In [ ]:
# Mount Google Drive and setup directories
drive.mount('/content/drive')

# Create necessary directories
!mkdir -p /content/utils
!mkdir -p /content/annotations
!mkdir -p /content/train2014
!mkdir -p /content/val2014
!mkdir -p /content/drive/MyDrive/show_and_tell/checkpoints

In [ ]:
# Download and setup MSCOCO dataset
def download_file(url, filename):
    """Download file with progress bar"""
    with tqdm(unit='B', unit_scale=True, unit_divisor=1024, miniters=1, desc=filename) as t:
        urllib.request.urlretrieve(
            url, filename,
            reporthook=lambda count, block_size, total_size: t.update(block_size)
        )

def setup_mscoco():
    urls = {
        'annotations': 'http://images.cocodataset.org/annotations/annotations_trainval2014.zip',
        'train': 'http://images.cocodataset.org/zips/train2014.zip',
        'val': 'http://images.cocodataset.org/zips/val2014.zip'
    }
    
    for name, url in urls.items():
        zip_file = f'/content/{name}.zip'
        if not os.path.exists(zip_file):
            print(f'Downloading {name} dataset...')
            download_file(url, zip_file)
            print(f'Extracting {name} dataset...')
            !unzip -q {zip_file}
            # Remove zip file to save space
            !rm {zip_file}

setup_mscoco()

In [ ]:
# Image preprocessing functions
IMAGE_SIZE = 299
attention_features_shape = 64  # For the attention mechanism

def load_image(image_path):
    img = load_img(image_path, target_size=(IMAGE_SIZE, IMAGE_SIZE))
    img = img_to_array(img)
    img = tf.keras.applications.vgg16.preprocess_input(img)
    return img

def map_func(img_path, cap):
    img = tf.io.read_file(img_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, (IMAGE_SIZE, IMAGE_SIZE))
    img = tf.keras.applications.vgg16.preprocess_input(img)
    return img, cap

In [ ]:
# Load and preprocess captions
def load_captions_data(filename):
    with open(filename, 'r') as f:
        annotations = json.load(f)
    
    all_captions = []
    all_img_paths = []

    for annot in annotations['annotations']:
        caption = f"<start> {annot['caption']} <end>"
        image_id = annot['image_id']
        img_path = f"/content/train2014/COCO_train2014_{int(image_id):012d}.jpg"
        
        if os.path.exists(img_path):
            all_captions.append(caption)
            all_img_paths.append(img_path)
    
    return all_captions, all_img_paths

# Load captions
captions, img_paths = load_captions_data('/content/annotations/captions_train2014.json')

# Create tokenizer
tokenizer = Tokenizer(num_words=5000, oov_token="<unk>", filters='!"#$%&()*+.,-/:;=?@[\\]^_`{|}~')
tokenizer.fit_on_texts(captions)

# Create word-to-index and index-to-word mappings
word_to_index = tokenizer.word_index
index_to_word = dict([(index, word) for word, index in word_to_index.items()])

# Pad sequences
cap_vector = tokenizer.texts_to_sequences(captions)
cap_vector = pad_sequences(cap_vector, padding='post')

In [ ]:
# Modern implementation of the Show and Tell model using Keras
class ShowAndTellModel(tf.keras.Model):
    def __init__(self, vocab_size, max_length, embedding_dim=512, units=512):
        super(ShowAndTellModel, self).__init__()
        
        # Load pretrained VGG16 without top layers
        self.cnn = VGG16(include_top=False, weights='imagenet')
        self.cnn.trainable = False
        
        # Feature extraction layers
        self.features_extract = tf.keras.Sequential([
            layers.GlobalAveragePooling2D(),
            layers.Dense(embedding_dim)
        ])
        
        # Caption generation layers
        self.embedding = layers.Embedding(vocab_size, embedding_dim)
        self.lstm = layers.LSTM(units, return_sequences=True, return_state=True)
        self.dense = layers.Dense(vocab_size)
        
        # Initialize states
        self.reset_state = lambda batch_size: tf.zeros((batch_size, units))
        
    def call(self, inputs):
        image, captions = inputs
        
        # Extract image features
        features = self.cnn(image)
        features = self.features_extract(features)
        
        # Embed captions
        x = self.embedding(captions)
        
        # LSTM with attention
        output, state_h, state_c = self.lstm(x, initial_state=[features, features])
        
        # Generate word probabilities
        x = self.dense(output)
        
        return x
        
    def decode_step(self, dec_input, features, hidden):
        # Embedding
        x = self.embedding(dec_input)
        
        # LSTM step
        output, state_h, state_c = self.lstm(x, initial_state=[hidden, features])
        
        # Generate prediction
        x = self.dense(output)
        
        return x, state_h, None  # None for attention weights as we're not using attention

In [ ]:
# Modern data pipeline using tf.data
def create_dataset(image_paths, captions, batch_size=32):
    dataset = tf.data.Dataset.from_tensor_slices((image_paths, captions))
    dataset = dataset.map(lambda item1, item2: tf.numpy_function(
          map_func, [item1, item2], [tf.float32, tf.int32]),
          num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.shuffle(1000)
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset

# Create training dataset
train_dataset = create_dataset(img_paths, cap_vector, config.batch_size)

In [ ]:
# Training configuration using modern practices
class Config:
    def __init__(self):
        self.batch_size = 32
        self.embedding_dim = 512
        self.units = 512
        self.vocab_size = len(word_to_index) + 1
        self.max_length = max(len(t) for t in cap_vector)
        self.learning_rate = 0.001
        
        # Use Colab paths
        self.train_image_dir = '/content/train2014/'
        self.val_image_dir = '/content/val2014/'
        self.checkpoint_path = '/content/drive/MyDrive/show_and_tell/checkpoints/'

config = Config()

In [ ]:
# Modern training loop with tf.keras
@tf.function
def train_step(model, optimizer, images, captions, loss_function):
    loss = 0
    
    with tf.GradientTape() as tape:
        predictions = model([images, captions[:, :-1]])
        loss = loss_function(captions[:, 1:], predictions)

    trainable_variables = model.trainable_variables
    gradients = tape.gradient(loss, trainable_variables)
    optimizer.apply_gradients(zip(gradients, trainable_variables))
    
    return loss

In [ ]:
# Initialize model and optimizer
model = ShowAndTellModel(config.vocab_size, config.max_length,
                        config.embedding_dim, config.units)
optimizer = tf.keras.optimizers.Adam(config.learning_rate)
loss_function = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=True, reduction='none')

# Checkpoint manager for saving models
ckpt = tf.train.Checkpoint(model=model, optimizer=optimizer)
ckpt_manager = tf.train.CheckpointManager(ckpt, config.checkpoint_path, max_to_keep=5)

In [ ]:
# Training loop
EPOCHS = 20

for epoch in range(EPOCHS):
    total_loss = 0
    
    for (batch, (img_tensor, target)) in enumerate(train_dataset):
        batch_loss = train_step(model, optimizer, img_tensor, target, loss_function)
        total_loss += batch_loss
        
        if batch % 100 == 0:
            print(f'Epoch {epoch+1} Batch {batch} Loss {batch_loss.numpy():.4f}')
    
    # Save checkpoint every 5 epochs
    if (epoch + 1) % 5 == 0:
        ckpt_manager.save()
        
    print(f'Epoch {epoch+1} Loss {total_loss/len(train_dataset):.6f}')

In [ ]:
# Modern caption generation function
def generate_caption(image_path):
    attention_plot = np.zeros((config.max_length, attention_features_shape))
    
    hidden = model.reset_state(batch_size=1)
    temp_input = tf.expand_dims(load_image(image_path), 0)
    img_tensor_val = model.cnn(temp_input)
    img_tensor_val = model.features_extract(img_tensor_val)
    
    dec_input = tf.expand_dims([word_to_index['<start>']], 0)
    result = []

    for i in range(config.max_length):
        predictions, hidden, _ = model.decode_step(
            dec_input, img_tensor_val, hidden)
        
        predicted_id = tf.random.categorical(predictions, 1)[0][0].numpy()
        word = index_to_word.get(predicted_id, '<unk>')
        result.append(word)
        
        if word == '<end>':
            break
            
        dec_input = tf.expand_dims([predicted_id], 0)
    
    return ' '.join(result)

# Example usage
# caption = generate_caption('path_to_your_image.jpg')
# print(f'Generated caption: {caption}')